# 🚢 Maritime VHF ASR — Dissertation Notebook
**Transformer-Based ASR Fine-Tuning on Real and Simulated Maritime Speech**

---
### Before you start
1. **Runtime → Change runtime type → T4 GPU**
2. Have your **Label Studio API key** ready → https://app.heartex.com/user/account

| Model | GPU needed |
|---|---|
| whisper-small / wav2vec2 | T4 (free tier) |
| whisper-medium | T4 (~30 min / experiment) |
| whisper-large + LoRA | A100 (Pro+) |


## ✅ Step 1 — Check GPU

In [ ]:
import torch
if torch.cuda.is_available():
    gpu = torch.cuda.get_device_properties(0)
    print(f"✅  GPU : {gpu.name}")
    print(f"   VRAM: {gpu.total_memory/1e9:.1f} GB")
else:
    print("❌  No GPU — Runtime → Change runtime type → T4 GPU")


## 📁 Step 2 — Mount Drive & Upload Project

In [ ]:
from google.colab import drive, files
import os, zipfile

drive.mount('/content/drive')
DRIVE_PROJECT = '/content/drive/MyDrive/ASR_Dissertation'
os.makedirs(DRIVE_PROJECT, exist_ok=True)
print(f"✅  Drive folder: {DRIVE_PROJECT}")


In [ ]:
print("📤  Select asr_dissertation_project.zip ...")
uploaded = files.upload()
zip_name = list(uploaded.keys())[0]

with zipfile.ZipFile(zip_name, 'r') as z:
    z.extractall('/content')

os.chdir('/content/asr_dissertation')
print("✅  Extracted. Working directory:", os.getcwd())


## 📦 Step 3 — Install Dependencies

Run **Cell A**, then **Runtime → Restart session**, then run **Cell B** and **Cell C**.
Do NOT re-run Cell A after restarting.


In [ ]:
# ── Cell A: install everything in one shot, then restart the session ──
!pip install -q \
    "scipy>=1.14.0" \
    "scikit-learn>=1.6.0" \
    transformers peft datasets accelerate evaluate \
    jiwer librosa soundfile omegaconf rich inflect \
    tensorboard seaborn
print("✅  Done — Runtime → Restart session now")


In [ ]:
# ── Cell B: run immediately after restart ──
import os
os.chdir('/content/asr_dissertation')
print("📂  Working directory:", os.getcwd())


In [ ]:
# ── Cell C: verify imports ──
import numpy, scipy, transformers, peft, omegaconf, evaluate
print(f"numpy        : {numpy.__version__}")
print(f"scipy        : {scipy.__version__}")       # must be 1.14+
print(f"transformers : {transformers.__version__}")
print(f"peft         : {peft.__version__}")
print("✅  All imports OK")


## 🔧 Step 4 — Apply Code Patches

Three bug-fixes that must be applied once per session (safe to re-run).


In [ ]:
# Patch 1 — export_tasks: use paginated /api/tasks endpoint
# (the /export endpoint is not available on Heartex cloud)
import re, pathlib

fp = pathlib.Path('/content/asr_dissertation/label_studio_export.py')
src = fp.read_text()

MARKER = 'def export_tasks'
if MARKER in src and 'while True' not in src:
    # Replace the method body up to the return statement
    old = (
        '    def export_tasks(self, project_id: int, export_type: str = "JSON") -> list[dict]:\n'
        '        """Export all completed tasks as JSON."""\n'
        '        resp = self._get(\n'
        '            f"/api/projects/{project_id}/export",\n'
        '            params={"exportType": export_type},\n'
        '            raw=True,\n'
        '        )\n'
        '        return json.loads(resp.content)'
    )
    new = (
        '    def export_tasks(self, project_id: int, export_type: str = "JSON") -> list[dict]:\n'
        '        """Fetch ALL tasks via paginated /api/tasks (Heartex cloud)."""\n'
        '        tasks, page = [], 1\n'
        '        while True:\n'
        '            resp = self._get(\n'
        '                "/api/tasks",\n'
        '                params={"project": project_id, "page": page,\n'
        '                        "page_size": 100, "fields": "all"},\n'
        '            )\n'
        '            if isinstance(resp, list):\n'
        '                tasks.extend(resp)\n'
        '                break\n'
        '            results = resp.get("tasks", resp.get("results", []))\n'
        '            if not results:\n'
        '                break\n'
        '            tasks.extend(results)\n'
        '            total = resp.get("total", resp.get("count", 0))\n'
        '            log.info("   Page %d: %d / %d", page, len(tasks), total)\n'
        '            if len(tasks) >= total:\n'
        '                break\n'
        '            page += 1\n'
        '        return tasks'
    )
    if old in src:
        fp.write_text(src.replace(old, new))
        print('✅  Patch 1 applied: export_tasks paginated')
    else:
        print('⚠️   Patch 1: old pattern not found — check label_studio_export.py')
else:
    print('ℹ️   Patch 1: already applied')


In [ ]:
# Patch 2 — download_file: use plain requests (no LS auth header for Azure SAS URLs)
import pathlib

fp = pathlib.Path('/content/asr_dissertation/label_studio_export.py')
src = fp.read_text()

if 'import requests as _requests' not in src:
    old = (
        '    def download_file(self, url: str, dest: Path) -> bool:\n'
        '        """Download a file, handling both absolute and relative URLs."""\n'
        '        if not url.startswith("http"):\n'
        '            url = self.base_url + url\n'
        '        try:\n'
        '            r = self.session.get(url, stream=True, timeout=60)'
    )
    new = (
        '    def download_file(self, url: str, dest: Path) -> bool:\n'
        '        """Download a file, handling both absolute and relative URLs."""\n'
        '        if not url.startswith("http"):\n'
        '            url = self.base_url + url\n'
        '        try:\n'
        '            import requests as _requests  # plain — no LS auth header\n'
        '            r = _requests.get(url, stream=True, timeout=60)'
    )
    if old in src:
        fp.write_text(src.replace(old, new))
        print('✅  Patch 2 applied: download_file uses plain requests')
    else:
        print('⚠️   Patch 2: old pattern not found — check label_studio_export.py')
else:
    print('ℹ️   Patch 2: already applied')


In [ ]:
# Patch 3 — WhisperDataCollator: stack audio features instead of tokenizer.pad()
# Whisper input_features are already 80x3000 — tokenizer.pad() only handles text
import pathlib

fp = pathlib.Path('/content/asr_dissertation/src/dataset.py')
src = fp.read_text()

if 'torch.stack([f["input_features"]' not in src:
    old = (
        '        batch = self.tokenizer.pad(input_features, return_tensors="pt")\n'
        '        labels_batch = self.tokenizer.pad(\n'
        '            label_features, padding=True, return_tensors="pt"\n'
        '        )\n'
        '\n'
        '        labels = labels_batch["input_ids"].masked_fill(\n'
        '            labels_batch.attention_mask.ne(1), -100\n'
        '        )\n'
        '        # Whisper SOS token should not be predicted\n'
        '        if (labels[:, 0] == self.tokenizer.bos_token_id).all():\n'
        '            labels = labels[:, 1:]\n'
        '\n'
        '        batch["labels"] = labels\n'
        '        batch["transcriptions"] = [f["transcription"] for f in features]\n'
        '        return batch'
    )
    new = (
        '        # input_features are fixed 80x3000 — stack, do not pad via tokenizer\n'
        '        batch = {\n'
        '            "input_features": torch.stack([f["input_features"] for f in features])\n'
        '        }\n'
        '        labels_batch = self.tokenizer.pad(\n'
        '            label_features, padding=True, return_tensors="pt"\n'
        '        )\n'
        '        labels = labels_batch["input_ids"].masked_fill(\n'
        '            labels_batch.attention_mask.ne(1), -100\n'
        '        )\n'
        '        if (labels[:, 0] == self.tokenizer.bos_token_id).all():\n'
        '            labels = labels[:, 1:]\n'
        '        batch["labels"] = labels\n'
        '        batch["transcriptions"] = [f["transcription"] for f in features]\n'
        '        return batch'
    )
    if old in src:
        fp.write_text(src.replace(old, new))
        print('✅  Patch 3 applied: WhisperDataCollator fixed')
    else:
        print('⚠️   Patch 3: old pattern not found — check src/dataset.py')
else:
    print('ℹ️   Patch 3: already applied')


## 🔑 Step 5 — Label Studio API Key & Download Data

In [ ]:
import os
try:
    from google.colab import userdata
    api_key = userdata.get('LABEL_STUDIO_API_KEY')
    print("✅  API key loaded from Colab Secrets")
except Exception:
    api_key = ""   # ← paste key here if not using Secrets
    if api_key:
        print("✅  API key set manually")
    else:
        print("⚠️   No API key — add to Colab Secrets (sidebar 🔑) as LABEL_STUDIO_API_KEY")

os.environ['LABEL_STUDIO_API_KEY'] = api_key


In [ ]:
# Download real dataset (Maritime_ASR_Main, project 185598)
# Simulated dataset SAS tokens expired — see Step 12 to re-enable
!python label_studio_export.py --api-key $LABEL_STUDIO_API_KEY --project real
# Expected: ✅ [real] Exported=1500+  Failed=0


## 🔧 Step 6 — Prepare Datasets

In [ ]:
# Validate audio, normalise transcriptions, create 80/10/10 train/val/test splits
!python main.py --mode data
# Expected: train ~1244 | val ~155 | test ~157


In [ ]:
import json, glob
for path in sorted(glob.glob('data/*/*.json')):
    with open(path) as f:
        n = len(json.load(f))
    print(f"{path:<50} {n:>5} samples")


## 🏋️ Step 7 — Training

**Start with this to verify your setup works (~30 min on T4):**
`ef_whisper_small_real`

Then run the full batch cell below.


In [ ]:
!python main.py --mode list

In [ ]:
# ── Single experiment ──
EXPERIMENT = "ef_whisper_small_real"
!python main.py --mode train --experiment {EXPERIMENT}


### Run all real-data experiments

In [ ]:
# Runs sequentially; already-finished experiments are skipped automatically.
# After a session reset: restore checkpoints (Step 8 RESTORE), then re-run.

EXPERIMENTS = [
    # Encoder freezing
    "ef_whisper_small_real",
    "ef_whisper_medium_real",
    "ef_wav2vec2_real",
    # LoRA (fixed: expanded target_modules, LR=1e-3, no 8-bit quant)
    "lora_whisper_small_real",
    "lora_whisper_medium_real",
    # ── Uncomment for A100 (Pro+) ──────────────────────────────────────
    # "ef_whisper_large_real",
    # "lora_whisper_large_real",
    # "full_ft_whisper_medium_real",
    # "full_ft_whisper_large_real",
]

import subprocess, sys
for exp in EXPERIMENTS:
    print(f"\n{'='*55}\n  {exp}\n{'='*55}")
    r = subprocess.run(
        [sys.executable, "main.py", "--mode", "train", "--experiment", exp],
        cwd="/content/asr_dissertation",
    )
    if r.returncode != 0:
        print(f"  ⚠️  {exp} failed — continuing...")


## 💾 Step 8 — Save / Restore Checkpoints from Drive

In [ ]:
# SAVE — run after each training session
import shutil, os

src = '/content/asr_dissertation/checkpoints'
dst = f'{DRIVE_PROJECT}/checkpoints'
if os.path.exists(src):
    shutil.copytree(src, dst, dirs_exist_ok=True)
    print(f"✅  Checkpoints saved to {dst}")
else:
    print("No checkpoints yet — run training first")


In [ ]:
# RESTORE — run at the start of a new session (after Step 3 Cell B)
import shutil

src = f'{DRIVE_PROJECT}/checkpoints'
dst = '/content/asr_dissertation/checkpoints'
if os.path.exists(src):
    shutil.copytree(src, dst, dirs_exist_ok=True)
    print("✅  Checkpoints restored from Drive")
else:
    print("No saved checkpoints on Drive yet")


## 📊 Step 9 — Evaluate All Experiments

In [ ]:
!python main.py --mode eval_all

In [ ]:
# Evaluate a specific checkpoint on the test set
CHECKPOINT = "checkpoints/ef_whisper_small_real"
MANIFEST   = "data/real/test_manifest.json"
!python main.py --mode test_wer --checkpoint {CHECKPOINT} --manifest {MANIFEST}


## 📈 Step 10 — Generate Dissertation Figures

In [ ]:
!python main.py --mode figures

import glob
for f in sorted(glob.glob('results/figures/*.png')):
    print(f)


In [ ]:
from IPython.display import Image, display
import glob

for fig in sorted(glob.glob('results/figures/*.png')):
    print(f"\n── {fig.split('/')[-1]} ──")
    display(Image(fig, width=900))


In [ ]:
import shutil
shutil.copytree('/content/asr_dissertation/results',
                f'{DRIVE_PROJECT}/results', dirs_exist_ok=True)
print(f"✅  Results saved to Drive")


## 📋 Step 11 — Results Summary Table

In [ ]:
import json, pandas as pd

try:
    with open('results/all_results.json') as f:
        all_results = json.load(f)

    rows = []
    for exp_name, res in all_results.items():
        for key, val in res.items():
            if key.startswith('eval_') and isinstance(val, dict) and 'wer' in val:
                rows.append({
                    'Experiment': exp_name,
                    'Method'    : res.get('method', ''),
                    'Train Data': str(res.get('train_data', '—')),
                    'Eval Set'  : key.replace('eval_', ''),
                    'WER (%)'   : round(val['wer'] * 100, 2),
                    'CER (%)'   : round(val.get('cer', 0) * 100, 2),
                })

    df = pd.DataFrame(rows).sort_values(['Method', 'Eval Set', 'WER (%)'])
    pd.set_option('display.max_rows', 100)
    print(df.to_string(index=False))

except FileNotFoundError:
    print("No results yet — run: python main.py --mode eval_all")


## 🚧 Step 12 — Simulated Data (when Azure access is restored)

The `sim_vhf_dataset` SAS tokens expired Feb 2026 and the Azure Blob Storage
connection was removed from Label Studio.

**To re-enable:**
1. Email **Derek Flanagan** (flanagan.derek@ul.ie) — ask him to reconnect
   `simvhf.blob.core.windows.net` to Label Studio project 226738
2. Once reconnected, update Step 5:
   ```
   --project both
   ```
3. Re-run Steps 6–11 to add the simulated / combined experiments:
   - `ef_whisper_small_simulated` / `ef_whisper_small_combined`
   - `lora_whisper_small_simulated` / `lora_whisper_small_combined`
   - etc.


## 🎙️ Bonus — Transcribe a Single File

In [ ]:
from src.inference import build_transcriber
from google.colab import files

print("Upload an audio file (WAV / MP3)...")
uploaded = files.upload()
audio_path = list(uploaded.keys())[0]

CHECKPOINT = "checkpoints/ef_whisper_small_real"   # ← change to best model

transcriber = build_transcriber("whisper", CHECKPOINT)
text, elapsed = transcriber.transcribe_file(audio_path)

print(f"\n📝  Transcription : {text}")
print(f"⏱️   Latency        : {elapsed*1000:.0f} ms")


---
## 🔄 Session Reset Checklist

After Colab disconnects or runtime restarts:

1. ✅ Step 1 — GPU check
2. 📁 Step 2 — mount Drive, re-upload zip
3. 📦 Step 3 — Cell A → restart → Cell B → Cell C
4. 🔧 **Step 4 — apply all 3 patches** (required every session)
5. 🔧 Step 6 — re-run data pipeline (fast, ~30s)
6. 💾 **Step 8 RESTORE** — pull checkpoints from Drive
7. 🏋️ Step 7 / Step 9 — continue training or go straight to eval

---
## ⚠️ Out of Memory?

Reduce batch size in `configs/config.yaml`:
```yaml
per_device_train_batch_size: 8   # reduce from 16
gradient_accumulation_steps: 4   # keep effective batch = 32
dataloader_num_workers: 2        # reduce from 4
fp16: true                       # already enabled for T4
```
